In [1]:
from langchain_ollama import ChatOllama

MODEL_NAME1 = "qwen3:8b"
MODEL_NAME2 = "mistral:7b"
MODEL_NAME3 = "llama3.1:8b"
OLLAMA_URL = "http://127.0.0.1:11434"

llm = ChatOllama(
    model=MODEL_NAME3,
    base_url=OLLAMA_URL,
    temperature=0,
)

print(f"Connected to Ollama. Using local model: {MODEL_NAME3}")

/home/kanadmarick/Workspace/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Connected to Ollama. Using local model: llama3.1:8b


In [2]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, interrupt
from typing_extensions import TypedDict
import os
from dotenv import load_dotenv
for env_path in [
    Path.cwd() / ".env",
    Path.cwd().parent / ".env",
    Path.cwd() / "Langgraph_tools" / "chatbot" / ".env",
]:
    if env_path.exists():
        load_dotenv(env_path, override=True)
        break

# Optional: warn instead of crashing if Tavily is not configured yet.
tavily_key = os.getenv("TAVILY_API_KEY")
print("TAVILY_API_KEY loaded:", bool(tavily_key))

# Minimal smoke test: confirm the model is actually responding.
response = llm.invoke("Reply with one short sentence confirming the model is working.")
print(response.content)

TAVILY_API_KEY loaded: True
The model is functioning as intended.


In [3]:
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"



In [10]:

class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
## Graph with tool call
from langchain_ollama import ChatOllama
from langchain_core.tools import tool

@tool
def add(a: float, b: float):
    """Add two numbers."""
    return a + b


tool_node = ToolNode([add])
llm_with_tool = llm.bind_tools([add])


def call_llm_model(state: State):
    messages = state["messages"]
    response = llm_with_tool.invoke(messages)
    return {"messages": [response]}


builder = StateGraph(State)
builder.add_node("call_llm_model", call_llm_model)
builder.add_node("tools", tool_node)

builder.add_edge(START, "call_llm_model")
builder.add_conditional_edges(
    "call_llm_model",
    tools_condition,
    {
        "tools": "tools",
        END: END,
    },
)
builder.add_edge("tools", "call_llm_model")

## COMPILE THE GRAPH
graph = builder.compile()

from IPython.display import display, Image

try:
    display(Image(filename="graph.png"))
except Exception as e:
    print("Graph image preview could not be displayed:", e)


ValueError: At 'call_llm_model' node, 'tools_condition' branch found unknown target 'tools'